# Silver Layer: Michelin Restaurant Data

This notebook transforms the raw Bronze dataset into a cleaned and standardized dataset for analysis.

Silver-layer transformations include:
- Standardizing column names
- Creating a deterministic restaurant identifier
- Standardizing price levels
- Creating Michelin star metrics
- Creating analytical indicator fields
- Handling placeholder values
- Preserving original source values where appropriate


In [0]:
SELECT
    Name AS restaurant_name,
    Address AS address,
    Location AS location,
    Latitude AS latitude,
    Longitude AS longitude,

    Cuisine AS cuisine,

    Price AS price_raw,

    CASE
        WHEN Price IN ('$', '€') THEN 1
        WHEN Price IN ('$$', '€€', '¥¥', '฿฿') THEN 2
        WHEN Price IN ('$$$', '€€€', '¥¥¥') THEN 3
        WHEN Price IN ('$$$$', '€€€€') THEN 4
        ELSE NULL
    END AS price_tier,

    Award AS award,

    CASE
        WHEN Award = '1 Star' THEN 1
        WHEN Award = '2 Stars' THEN 2
        WHEN Award = '3 Stars' THEN 3
        ELSE 0
    END AS star_count,

    CASE
        WHEN Award IN ('1 Star', '2 Stars', '3 Stars') THEN TRUE
        ELSE FALSE
    END AS is_starred,

    CASE
        WHEN GreenStar = 1 THEN TRUE
        ELSE FALSE
    END AS is_green_star,

    PhoneNumber AS phone_number,
    Url AS michelin_url,
    WebsiteUrl AS website_url,
    FacilitiesAndServices AS facilities_and_services,
    Description AS description

FROM workspace.michelin_analytics.bronze_michelin_restaurants
LIMIT 20;

restaurant_name,address,location,latitude,longitude,cuisine,price_raw,price_tier,award,star_count,is_starred,is_green_star,phone_number,michelin_url,website_url,facilities_and_services,description
8 1/2 Otto e Mezzo - Bombana,"Shop 202, 2F, Alexandra House, 18 Chater Road, Central, Hong Kong, Hong Kong SAR China","Hong Kong, Hong Kong SAR China",22.2814635,114.15867,Italian,$$$$,4,3 Stars,3,true,false,85225378859,https://guide.michelin.com/en/hong-kong-region/hong-kong/restaurant/8%C2%BD-otto-e-mezzo-bombana,https://www.ottoemezzobombana.com,"Air conditioning,Car park,Interesting wine list,Valet parking,Wheelchair access","(The restaurant is temporarily closed.) Owner-chef Umberto Bombana’s restaurant, which oozes Italian charm and passion from every pore, is named after one of the chef’s favourite Fellini films. The classical Italian lineup stars first-class ingredients, like lamb from Aveyron, scallops from Hokkaido and Wagyu beef from Australia. The pasta course is always a highlight – scampi risotto and trenette with sea urchin are just two of the standouts. The delightful service makes everyone feel special."
ABaC,"Avenida del Tibidabo 1, Barcelona, 08022, Spain","Barcelona, Spain",41.4104497,2.1365112,Creative,$$$$,4,3 Stars,3,true,false,34933196600,https://guide.michelin.com/en/catalunya/barcelona/restaurant/abac,https://abacrestaurant.com,"Air conditioning,Interesting wine list,Terrace","Tradition, cutting-edge, flavour, attitude and passion, along with a consistent approach, define to perfection the cooking of Jordi Cruz. In his elegant restaurant boasting views of a tranquil garden dotted with some amazing works of contemporary art, he offers diners a unique gastronomic experience that constantly evolves in line with ingredients from the changing seasons. These are always combined with his technical ability as he revisits Catalan and other Mediterranean flavours with a nod to influences from elsewhere around the globe. The experience here, which begins with appetisers in the kitchen, is centred around a single tasting menu featuring new recipes alongside the restaurant’s renowned signature dishes. One of the most extraordinary dishes is the Aqua Mirabilis dessert, an original tribute to Johann Maria Farina, who, in 1709, founded the oldest perfume factory in the world. This course, which plays with different textures, allows you to taste a scent through a wonderful sensory game."
AM par Alexandre Mazzia,"9 rue François-Rocca, Marseille, 13008, France","Marseille, France",43.2701104,5.3862325,Creative,$$$$,4,3 Stars,3,true,false,33491248363,https://guide.michelin.com/en/provence-alpes-cote-dazur/marseille/restaurant/am-par-alexandre-mazzia,https://www.alexandre-mazzia.com/,"Air conditioning,Interesting wine list,Pets allowed everywhere","In his eatery near the Stade Vélodrome, this artist-cum-chef on a perpetual creative quest has elevated the small portion to the realm of art. He is a virtuoso when it comes to spices, roasting and smoking and his childhood in Congo flows freely through his cuisine. This distinctive hallmark is further enriched by a unique use of vinegars (over 400 house creations) that add depth and complexity to his jus. The resulting lineup of some forty small dishes, a truly herculean task, is surgically crafted and often visually striking: cuttlefish steamed in sake, puffed barley, briny seasoning and seaweed; watercress, smoked beetroot, langoustine milk, iced burnt langoustine heads. Each meal is a feat of culinary excellence, supervised by a diffident, yet single-minded chef, whose watchful eye never leaves the serving hatch."
Addison,"Fairmont Grand Del Mar, 5200 Grand Del Mar Way, San Diego, CA, 92130, USA","San Diego, CA, USA",32.9412972,-117.1988908,Contemporary,$$$$,4,3 Stars,3,true,false,18583141900,https://guide.michelin.com/en/california/us-san-diego/restaurant/addison,https://www.addisondelmar.com/,"Air conditioning,Car park,Garden or park,Interesting wine list,Terrace,Valet parking,W

### Create Unique ID

In [0]:
SELECT
    SHA2(
        CONCAT_WS(
            '|',
            TRIM(Name),
            TRIM(Address),
            TRIM(Location),
            CAST(Latitude AS STRING),
            CAST(Longitude AS STRING)
        ),
        256
    ) AS restaurant_id,

    TRIM(Name) AS restaurant_name,
    TRIM(Address) AS address,
    TRIM(Location) AS location,
    Latitude AS latitude,
    Longitude AS longitude,

    TRIM(Cuisine) AS cuisine,

    Price AS price_raw,

    CASE
        WHEN Price IN ('$', '€') THEN 1
        WHEN Price IN ('$$', '€€', '¥¥', '฿฿') THEN 2
        WHEN Price IN ('$$$', '€€€', '¥¥¥') THEN 3
        WHEN Price IN ('$$$$', '€€€€') THEN 4
        ELSE NULL
    END AS price_tier,

    Award AS award,

    CASE
        WHEN Award = '1 Star' THEN 1
        WHEN Award = '2 Stars' THEN 2
        WHEN Award = '3 Stars' THEN 3
        ELSE 0
    END AS star_count,

    CASE
        WHEN Award IN ('1 Star', '2 Stars', '3 Stars') THEN TRUE
        ELSE FALSE
    END AS is_starred,

    CASE
        WHEN GreenStar = 1 THEN TRUE
        ELSE FALSE
    END AS is_green_star,

    PhoneNumber AS phone_number,
    Url AS michelin_url,
    WebsiteUrl AS website_url,
    FacilitiesAndServices AS facilities_and_services,
    Description AS description

FROM workspace.michelin_analytics.bronze_michelin_restaurants;

restaurant_id,restaurant_name,address,location,latitude,longitude,cuisine,price_raw,price_tier,award,star_count,is_starred,is_green_star,phone_number,michelin_url,website_url,facilities_and_services,description
5684c10c3fd8fa406aaeb8c12fc138358ef5a6629c100400edd5732ba93fab4d,8 1/2 Otto e Mezzo - Bombana,"Shop 202, 2F, Alexandra House, 18 Chater Road, Central, Hong Kong, Hong Kong SAR China","Hong Kong, Hong Kong SAR China",22.2814635,114.15867,Italian,$$$$,4,3 Stars,3,true,false,85225378859,https://guide.michelin.com/en/hong-kong-region/hong-kong/restaurant/8%C2%BD-otto-e-mezzo-bombana,https://www.ottoemezzobombana.com,"Air conditioning,Car park,Interesting wine list,Valet parking,Wheelchair access","(The restaurant is temporarily closed.) Owner-chef Umberto Bombana’s restaurant, which oozes Italian charm and passion from every pore, is named after one of the chef’s favourite Fellini films. The classical Italian lineup stars first-class ingredients, like lamb from Aveyron, scallops from Hokkaido and Wagyu beef from Australia. The pasta course is always a highlight – scampi risotto and trenette with sea urchin are just two of the standouts. The delightful service makes everyone feel special."
7e7444d309ce34955ebc87ac74da9df7f74df156d10175f97223e95bc8e478bb,ABaC,"Avenida del Tibidabo 1, Barcelona, 08022, Spain","Barcelona, Spain",41.4104497,2.1365112,Creative,$$$$,4,3 Stars,3,true,false,34933196600,https://guide.michelin.com/en/catalunya/barcelona/restaurant/abac,https://abacrestaurant.com,"Air conditioning,Interesting wine list,Terrace","Tradition, cutting-edge, flavour, attitude and passion, along with a consistent approach, define to perfection the cooking of Jordi Cruz. In his elegant restaurant boasting views of a tranquil garden dotted with some amazing works of contemporary art, he offers diners a unique gastronomic experience that constantly evolves in line with ingredients from the changing seasons. These are always combined with his technical ability as he revisits Catalan and other Mediterranean flavours with a nod to influences from elsewhere around the globe. The experience here, which begins with appetisers in the kitchen, is centred around a single tasting menu featuring new recipes alongside the restaurant’s renowned signature dishes. One of the most extraordinary dishes is the Aqua Mirabilis dessert, an original tribute to Johann Maria Farina, who, in 1709, founded the oldest perfume factory in the world. This course, which plays with different textures, allows you to taste a scent through a wonderful sensory game."
c8900b389c96cf16d10f5c4634f748d10ad0b93ac461f61ba6be3e57c07e5144,AM par Alexandre Mazzia,"9 rue François-Rocca, Marseille, 13008, France","Marseille, France",43.2701104,5.3862325,Creative,$$$$,4,3 Stars,3,true,false,33491248363,https://guide.michelin.com/en/provence-alpes-cote-dazur/marseille/restaurant/am-par-alexandre-mazzia,https://www.alexandre-mazzia.com/,"Air conditioning,Interesting wine list,Pets allowed everywhere","In his eatery near the Stade Vélodrome, this artist-cum-chef on a perpetual creative quest has elevated the small portion to the realm of art. He is a virtuoso when it comes to spices, roasting and smoking and his childhood in Congo flows freely through his cuisine. This distinctive hallmark is further enriched by a unique use of vinegars (over 400 house creations) that add depth and complexity to his jus. The resulting lineup of some forty small dishes, a truly herculean task, is surgically crafted and often visually striking: cuttlefish steamed in sake, puffed barley, briny seasoning and seaweed; watercress, smoked beetroot, langoustine milk, iced burnt langoustine heads. Each meal is a feat of culinary excellence, supervised by a diffident, yet single-minded chef, whose watchful eye never leaves the serving hatch."
1c16f60f4db9fd50a1ad802e44f5c7c740de3265388e96bb1247d12b7ceca7c1,Addison,"Fairmont Grand Del Mar, 5200 Grand Del Mar Way, San Diego, CA, 92130, USA","San Diego, CA,

Validate ID

In [0]:
WITH silver_preview AS (

    SELECT
        SHA2(
            CONCAT_WS(
                '|',
                TRIM(Name),
                TRIM(Address),
                TRIM(Location),
                CAST(Latitude AS STRING),
                CAST(Longitude AS STRING)
            ),
            256
        ) AS restaurant_id

    FROM workspace.michelin_analytics.bronze_michelin_restaurants
)

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT restaurant_id) AS unique_restaurant_ids,
    SUM(CASE WHEN restaurant_id IS NULL THEN 1 ELSE 0 END) AS null_restaurant_ids
FROM silver_preview;

total_rows,unique_restaurant_ids,null_restaurant_ids
19622,19622,0


## Create Silver Table

In [0]:
CREATE OR REPLACE TABLE workspace.michelin_analytics.silver_michelin_restaurants AS

SELECT
    SHA2(
        CONCAT_WS(
            '|',
            TRIM(Name),
            TRIM(Address),
            TRIM(Location),
            CAST(Latitude AS STRING),
            CAST(Longitude AS STRING)
        ),
        256
    ) AS restaurant_id,

    TRIM(Name) AS restaurant_name,
    TRIM(Address) AS address,
    TRIM(Location) AS location,

    Latitude AS latitude,
    Longitude AS longitude,

    TRIM(Cuisine) AS cuisine,

    Price AS price_raw,

    CASE
        WHEN Price IN ('$', '€') THEN 1
        WHEN Price IN ('$$', '€€', '¥¥', '฿฿') THEN 2
        WHEN Price IN ('$$$', '€€€', '¥¥¥') THEN 3
        WHEN Price IN ('$$$$', '€€€€') THEN 4
        ELSE NULL
    END AS price_tier,

    Award AS award,

    CASE
        WHEN Award = '1 Star' THEN 1
        WHEN Award = '2 Stars' THEN 2
        WHEN Award = '3 Stars' THEN 3
        ELSE 0
    END AS star_count,

    CASE
        WHEN Award IN ('1 Star', '2 Stars', '3 Stars') THEN TRUE
        ELSE FALSE
    END AS is_starred,

    CASE
        WHEN GreenStar = 1 THEN TRUE
        ELSE FALSE
    END AS is_green_star,

    PhoneNumber AS phone_number,
    Url AS michelin_url,
    WebsiteUrl AS website_url,
    FacilitiesAndServices AS facilities_and_services,
    Description AS description

FROM workspace.michelin_analytics.bronze_michelin_restaurants;

num_affected_rows,num_inserted_rows


### Validate it

In [0]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT restaurant_id) AS unique_restaurant_ids,
    SUM(CASE WHEN restaurant_id IS NULL THEN 1 ELSE 0 END) AS null_ids,
    SUM(CASE WHEN price_tier IS NULL THEN 1 ELSE 0 END) AS null_price_tiers,
    SUM(CASE WHEN star_count < 0 OR star_count > 3 THEN 1 ELSE 0 END) AS invalid_star_counts
FROM workspace.michelin_analytics.silver_michelin_restaurants;

total_rows,unique_restaurant_ids,null_ids,null_price_tiers,invalid_star_counts
19622,19622,0,1,0
